In [1]:
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
from moc.metrics.metrics_computer import compute_coverage_indicator, compute_log_region_size
from moc.conformal.conformalizers import L_CP, HDR_H
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import torch
import wandb

In [2]:
def proj_for(x_values, u, sample):
    sample_proj = torch.matmul(sample, u)
        
    if x_values is None:
        x_values = sample

    x_proj = torch.matmul(x_values, u)
    sample_sorted = torch.sort(sample_proj)[0]
    n = len(sample)
    
    cdf_values = torch.searchsorted(sample_sorted.T  , x_proj.unsqueeze(-1), side='right') / n
    return x_proj, cdf_values

def calculate_pit(values, u, sample):
        u = torch.as_tensor(u, dtype=sample.dtype, device=sample.device)
        values = torch.as_tensor(values, dtype=sample.dtype, device=sample.device)
        return proj_for(values, u, sample)[1]


#Fonction pour afficher les graphiques PIT
def plot_pit(pit_values, dimension, n_samples):

    cols = 3  # Fixe 3 graphes par ligne
    rows = int(np.ceil(dimension / cols))  # Nombre de lignes nécessaires

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))  # Ajustement dynamique de la taille

    # S'assurer que `axs` est toujours un tableau 2D pour éviter les erreurs
    axs = np.array(axs).reshape(rows, cols)

    for i in range(dimension):
        row, col = divmod(i, cols)  # Trouver la position dans la grille
        ax = axs[row, col]  # Récupérer l'axe correspondant
        
        ax.hist(pit_values[i], bins=10, density=True, alpha=0.6, color='g')
        title = f"PIT - Direction selon la composante ${i+1}$" 
        ax.set_title(title)
        ax.set_xlabel("Valeur projetée du PIT")
        ax.set_ylabel("Densité")

    # Supprimer les sous-graphiques vides si `dimension` n'est pas un multiple de 3
    for i in range(dimension, rows * cols):
        fig.delaxes(axs.flatten()[i])

    plt.tight_layout()  # Ajuster automatiquement l'affichage
    filename = f"PIT_proj_PCA_{n_samples}_VIctor.png" 
    plt.savefig(filename)
    plt.show()

In [3]:
config = get_config()
config.device = 'cpu'
#rc = RunConfig(config, 'mulan', 'rf2')
#rc = RunConfig(config,'feldman', 'bio')
rc = RunConfig(config,'camehl', 'households')
#rc = RunConfig(config,'del_barrio', 'ansur2')
datamodule = RealDataModule(rc)

In [4]:
datamodule.get_data()[0].shape

(7207, 14)

In [5]:
datamodule.get_data()[1].shape

(7207, 4)

In [5]:
p, q = datamodule.input_dim, datamodule.output_dim
print(p,q)

14 4


In [6]:
model = MixtureLightningModule(p,q)
#model = MQF2LightningModule(p, q)
trainer = get_lightning_trainer(rc)
trainer.fit(model, datamodule)
wandb.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: ryuzaki to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/elnurazhalieva/miniconda3/envs/scoring/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/elnurazhalieva/miniconda3/envs/scoring/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/Users/elnurazhalieva/miniconda3/envs/scoring/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


train_lambda_rqr,█▁▅▄▁▄▁▂▃▄▅▅▃▇▄▄▅▃▆▅▅▆▅▅▅▅▅▄█▅▆▅▅▄▇▃▆▇▇▆
train_loss,▆█▆▆▅▆▄▄▄▅▃▃▃▃▂▂▃▃▂▂▂▂▂▂▂▂▂▁▁▂▂▂▁▁▁▁▂▁▁▂
train_reg_loss,█▇▇▇▅▅▅▄▄▄▄▄▃▃▃▃▂▂▃▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▂
train_rqr,▅▂▂▅▂▁▂▁▂▂▄▃▂▄▂▄▅▄▃▅▄▄▅▅▄▆▅▃▇▄▆▃▄▅▅▄▃█▄▄
val_lambda_rqr,▄▂▃▃▁▅▃▅▃▆▆▄▅▇▅▄▇▅▅▅▇▇▇▇▅▇█▆█▇▆█▅▇██▆█▆▆
val_loss,▇█▇▆▆▅▄▄▄▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_reg_loss,▇█▇▆▆▅▅▄▄▂▂▂▂▂▂▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁
val_rqr,█▆▆▅▅▃▃▁▃▂▄▅▄▅▄▄▄▅▄▆▆▆▅▄▆▆▆▆▆▄▆▅▅▆▆▅▆▅▆▆
train_lambda_rqr,-8e-05
train_loss,2.24957
train_reg_loss,2.24949
